# Resolve the late-sampling win at 10 folds

The deployable change is **log@40 -> late@40** (S3_late vs S0_log): +0.0051, but **t 2.12** at
5 folds is right on the line, and YetiRank (t 1.45) already showed a sub-2 edge collapsing
on cloud. 10 folds doesn't change the true effect - it halves the estimate's noise. If the
effect is a stable +0.005, t should climb toward ~3; if it was a 5-fold fluke, t stays low.

Three schemes, ONE shared 10-fold split (seed 0), paired vs log:
- **S0_log** deployed baseline (log@40)
- **S3_late** ship candidate (late density @40) - the real deployable comparison
- **S_lean_late** late shape at log's row budget - mechanism (placement with rows held fixed)

In [1]:
import os, sys, json, time
import numpy as np, pandas as pd
from catboost import CatBoostClassifier

TOOLS = os.path.abspath('../tools')
sys.path.insert(0, TOOLS)
import cv_tools as CV

d = np.load(os.path.join(TOOLS, 'cv_folds_by_series', 'cv_cache_full.npz'))
X, y, sid, tim = d['X'], d['y'], d['sid'], d['time']
log_mask = d['is_sampled_step']
index = pd.MultiIndex.from_arrays([sid, tim], names=['id', 'time'])
step = CV.steps_from_index(index)
folds = CV.series_folds(sid, n_splits=10, seed=0)      # 10 folds now
uids, starts = np.unique(sid, return_index=True)
bounds = np.append(starts, len(sid)); lengths = np.diff(bounds)
print(f'{len(folds)} folds | val sizes {[len(b) for _, b in folds][:3]}...')

10 folds | val sizes [1000, 1000, 1000]...


In [2]:
def linear_late(L, m):
    if L <= m:
        return np.arange(L)
    u = np.linspace(0, 1, 512); cdf = np.cumsum(u); cdf /= cdf[-1]
    probs = (np.arange(m) + 0.5) / m
    return np.unique(np.round(np.interp(probs, cdf, u) * (L - 1)).astype(int))

def build(m):
    mask = np.zeros(len(sid), bool)
    for k in range(len(uids)):
        mask[bounds[k] + linear_late(lengths[k], m)] = True
    return mask

late40 = build(40)
# S_lean: reduce m until total rows <= log budget
target = int(log_mask.sum())
for mm in range(40, 20, -1):
    if sum(len(linear_late(L, mm)) for L in lengths) <= target:
        lean = build(mm); lean_m = mm; break
MASKS = {'S0_log': log_mask, 'S3_late': late40, 'S_lean_late': lean}
for n, m in MASKS.items():
    print(f'{n:16s} rows {m.sum():>8,} | pos-rate {y[m].mean():.3f}')
print(f'(S_lean m={lean_m})')

S0_log           rows  328,996 | pos-rate 0.111
S3_late          rows  393,759 | pos-rate 0.334
S_lean_late      rows  326,092 | pos-rate 0.335
(S_lean m=33)


In [3]:
CATP = dict(iterations=1500, learning_rate=0.02, depth=6, l2_leaf_reg=3.0,
            bootstrap_type='Bernoulli', subsample=0.8, loss_function='Logloss',
            random_seed=0, verbose=0, thread_count=-1)
def cat_fp(train, val):
    m = CatBoostClassifier(**CATP)
    m.fit(train.X, train.y, sample_weight=train.w)
    return m.predict_proba(val.X)[:, 1]

OUT = 'sampling_resolve_results.json'
done = json.load(open(OUT)) if os.path.exists(OUT) else {}
from cv_tools import CVResult
res = {}
for name, mask in MASKS.items():
    if name in done:
        res[name] = CVResult(np.array(done[name]['fold_scores']), None, done[name]['oof'], len(folds))
        print(f'{name}: cached ({res[name].mean:.4f})'); continue
    t = time.time()
    print(f'\n=== {name} (10 folds) ===', flush=True)
    r = CV.run_cv(X, y, sid, index, cat_fp, folds=folds, train_row_mask=mask,
                  row_step=step, verbose=False)
    res[name] = r
    done[name] = dict(fold_scores=r.fold_scores.tolist(), mean=r.mean, std=r.std,
                      sem=r.sem, oof=r.oof_score)
    json.dump(done, open(OUT, 'w'), indent=2)
    print(f'{r.summary(name)} | {time.time() - t:.0f}s', flush=True)


=== S0_log (10 folds) ===


S0_log             mean 0.6043 +/- 0.0154 (sem 0.0049) | OOF 0.6033 | folds: 0.6040  0.5774  0.5980  0.6141  0.5887  0.5973  0.6144  0.5998  0.6223  0.6272 | 572s



=== S3_late (10 folds) ===


S3_late            mean 0.6044 +/- 0.0151 (sem 0.0048) | OOF 0.6039 | folds: 0.6087  0.5861  0.5982  0.5990  0.5912  0.5959  0.6220  0.5933  0.6163  0.6331 | 670s



=== S_lean_late (10 folds) ===


S_lean_late        mean 0.6059 +/- 0.0150 (sem 0.0048) | OOF 0.6054 | folds: 0.6104  0.5880  0.5962  0.6040  0.5909  0.5938  0.6230  0.6003  0.6193  0.6328 | 580s


In [4]:
print(CV.summarize(res))
print('\n=== deployable: late@40 vs log@40 (the ship decision) ===')
print(CV.paired_compare(res['S3_late'], res['S0_log'], 'S3_late', 'S0_log'))
print('\n=== mechanism: placement at fixed row budget ===')
print(CV.paired_compare(res['S_lean_late'], res['S0_log'], 'S_lean_late', 'S0_log'))

S_lean_late        mean 0.6059 +/- 0.0150 (sem 0.0048) | OOF 0.6054 | folds: 0.6104  0.5880  0.5962  0.6040  0.5909  0.5938  0.6230  0.6003  0.6193  0.6328
S3_late            mean 0.6044 +/- 0.0151 (sem 0.0048) | OOF 0.6039 | folds: 0.6087  0.5861  0.5982  0.5990  0.5912  0.5959  0.6220  0.5933  0.6163  0.6331
S0_log             mean 0.6043 +/- 0.0154 (sem 0.0049) | OOF 0.6033 | folds: 0.6040  0.5774  0.5980  0.6141  0.5887  0.5973  0.6144  0.5998  0.6223  0.6272

=== deployable: late@40 vs log@40 (the ship decision) ===
S3_late - S0_log: +0.0001 +/- 0.0075 (sem 0.0024, t +0.03, wins 6/10) -> within noise
  per-fold deltas: +0.0048  +0.0087  +0.0002  -0.0151  +0.0024  -0.0014  +0.0076  -0.0064  -0.0061  +0.0059

=== mechanism: placement at fixed row budget ===
S_lean_late - S0_log: +0.0016 +/- 0.0064 (sem 0.0020, t +0.77, wins 6/10) -> within noise
  per-fold deltas: +0.0065  +0.0106  -0.0018  -0.0101  +0.0022  -0.0035  +0.0086  +0.0005  -0.0030  +0.0056
